### CHROMA DB DEMO 

- here we will build a RAG Question-Answer system using LLamaindex or Langchain
- Here the LLM is able to answer the user questions using the data provided by the user which is stored as an embedding in chromaDB

In [ ]:
import os
import csv
from dotenv import load_dotenv 

#chromaDB
import chromadb
from llama_index.core import Document

#llamaindex
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext
from llama_index.core import VectorStoreIndex
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.google_genai import GoogleGenAI

#langchain 
from langchain.schema.document import Document
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chains import RetrievalQA #query engine in llamaindex == langchain.chains in langchain
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

#RetrievalQA is the chain we used in langchain to ask ques and with the LLM. It retrieves relevant vectors from the database and then feed
#to the LLM and gets the response


In [3]:
# Load variables from .env file
load_dotenv()

True

In [4]:
# Access the keys
openai_key = os.getenv("OPENAI_API_KEY")
google_key = os.getenv("GOOGLE_API_KEY")

print("Keys loaded successfully")

Keys loaded successfully


- Preparing the data 
- same data source consisting of 14 articles from towards ai blog

In [ ]:
#!curl -o /Users/r31881/Desktop/AI/helloai/data/mini-dataset.csv https://raw.githubusercontent.com/AlaFalaki/tutorial_notebooks/main/data/mini-llama-articles.csv


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  169k  100  169k    0     0   288k      0 --:--:-- --:--:-- --:--:--  288k


- we will read the data csv parse through all the rows and then concatinate all the data together as a single string

In [5]:
text = ""
#load the JSON file 
with open("/Users/r31881/Desktop/AI/helloai/data/mini-llama-articles.csv", mode = "r", encoding = "utf-8") as file:
    csv_reader = csv.reader(file)

    for idx, row in enumerate(csv_reader):
        if idx == 0:
            continue
        text += row[1]



In [6]:
len(text)

171044

- CHUNKING 

In [7]:
chunk_size = 512 # this is a hyperparameter 
chunks = []

for i in range(0,len(text),chunk_size):
    chunks.append(text[i:i+chunk_size])
    

In [8]:
print(len(chunks))

335


In [9]:
chunks[0]

"LLM Variants and Meta's Open Source Before shedding light on four major trends, I'd share the latest Meta's Llama 2 and Code Llama. Meta's Llama 2 represents a sophisticated evolution in LLMs. This suite spans models pretrained and fine-tuned across a parameter spectrum of 7 billion to 70 billion. A specialized derivative, Llama 2-Chat, has been engineered explicitly for dialogue-centric applications. Benchmarking revealed Llama 2's superior performance over most extant open-source chat models. Human-centri"

- converting the chunks we have obtained so far into LLamaindex documents format

In [11]:
documents = [Document(text=t) for t in chunks]

In [ ]:
#consist information regarding the metadata and the chunk text itself
documents[:4]

[Document(id_='987c7ee5-2337-4075-9fc0-a8b52f987bea', embedding=None, metadata={}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text="LLM Variants and Meta's Open Source Before shedding light on four major trends, I'd share the latest Meta's Llama 2 and Code Llama. Meta's Llama 2 represents a sophisticated evolution in LLMs. This suite spans models pretrained and fine-tuned across a parameter spectrum of 7 billion to 70 billion. A specialized derivative, Llama 2-Chat, has been engineered explicitly for dialogue-centric applications. Benchmarking revealed Llama 2's superior performance over most extant open-source chat models. Human-centri", path=None, url=None, mimetype=None), image_resource=None, audio_resource=None, video_resource=None, text_template='{metadata_str}\n\n{content}'),
 Document(id_='e24879b1-1b85-4c22-963a-60542d962393

### Setting up CHROMA DB

In [17]:
#setting up chroma client
#PersistentClient saves the data at a desired location 
#EphemeralClient saves the data in memory
chroma_client = chromadb.PersistentClient(path = "/Users/r31881/Desktop/AI/helloai/data/mini-chunked-dataset")
chroma_collection = chroma_client.create_collection("mni-chunked-dataset")


In [19]:
#defining the vector store
vector_store = ChromaVectorStore(chroma_collection = chroma_collection)
storage_context = StorageContext.from_defaults(vector_store = vector_store)

- Loading the data into the vector database
- we will generate the index/generate embeddings using OpenAI embedding model 


In [21]:
index = VectorStoreIndex.from_documents(
    documents,
    embed_model = OpenAIEmbedding(model="text-embedding-3-small"),
    storage_context = storage_context,
    show_progress = True
)

Parsing nodes:   0%|          | 0/335 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/335 [00:00<?, ?it/s]

- Set up the query engine interface 
- It is responsible for retrieving related pieces of text, and using a LLM to formulate the final answer

In [31]:
llm = GoogleGenAI(model = "gemini-3.5-flash",max_tokens = 512, temperature = 0)
query_engine = index.as_query_engine(llm = llm, similarity_top_k = 5)

- getting the LLM response of our query 

In [32]:
response = query_engine.query("How many parameters LLaMA2 model has?")


Retrying llama_index.llms.google_genai.base.GoogleGenAI._chat in 1 seconds as it raised ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}.


In [33]:
print(response)

Based on the provided context, the Llama 2 model is available in four different parameter sizes: 
* 7 billion
* 13 billion
* 34 billion
* 70 billion


### Interface Chroma with LangChain

- convert chunks to the documents so that Lanchain framework can process them

In [36]:
documents = [Document(page_content = t) for t in chunks]

In [37]:
len(documents)

335

In [38]:
documents[:4]

[Document(metadata={}, page_content="LLM Variants and Meta's Open Source Before shedding light on four major trends, I'd share the latest Meta's Llama 2 and Code Llama. Meta's Llama 2 represents a sophisticated evolution in LLMs. This suite spans models pretrained and fine-tuned across a parameter spectrum of 7 billion to 70 billion. A specialized derivative, Llama 2-Chat, has been engineered explicitly for dialogue-centric applications. Benchmarking revealed Llama 2's superior performance over most extant open-source chat models. Human-centri"),
 Document(metadata={}, page_content="c evaluations, focusing on safety and utility metrics, positioned Llama 2-Chat as a potential contender against proprietary, closed-source counterparts. The development trajectory of Llama 2 emphasized rigorous fine-tuning methodologies. Meta's transparent delineation of these processes aims to catalyze community-driven advancements in LLMs, underscoring a commitment to collaborative and responsible AI deve

- create embeddings of these documents and store it in the chroma DB

In [47]:


# Add the documents to chroma DB and create Index / embeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

chroma_db = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    persist_directory="/Users/r31881/Desktop/AI/helloai/data/mini-chunked-dataset-chroma",
    collection_name="mini-chunked-dataset",
)

### Query Dataset

In [48]:
llm = ChatGoogleGenerativeAI(
    model = "gemini-3.5-flash",
    temperature = 0,
    max_tokens = 512,
)

In [50]:


query = "How many parameters LLaMA 2 model has?"

retriever = chroma_db.as_retriever(search_kwargs={"k": 4})
# Define a RetrievalQA chain that is responsible for retrieving related pieces of text,
# and using a LLM to formulate the final answer.
chain = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=retriever)

response = chain.invoke(query)
print(response["result"])

Based on the provided text, Llama 2 is available in four different parameter sizes: 
* 7 billion
* 13 billion
* 34 billion
* 70 billion
